# Mini project 3 — Person 1: Default simulation and collateral cash flows

This notebook simulates **1,000 scenarios for 10 bonds**, converts default times to quarterly cash flows, and sums the collateral available each quarter. Run all cells from top to bottom. Change `SCENARIO_ID` to inspect any case from 1 to 1,000.

**Conventions:** Dollar amounts are in USD; times are in years. Each bond pays a $150,000 quarterly coupon. On default, it pays recovery of 40% of face value ($4 million) once, at the end of the quarter containing default. It pays no coupon in that quarter and nothing afterward. A default exactly on a payment date takes precedence over that payment. There is no accrued coupon. The document specifies LGD but does not specify precise recovery timing, so these are explicit modeling assumptions.

**Correlation requirement:** Every pair of uncensored default times has a target population Pearson correlation of **0.20**. We retain the document's common-factor construction, but calibrate its normal-score correlation parameter to meet this requirement after the transformation to default times. The 1,000-scenario sample correlations will fluctuate around the population target.

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import ndtr, log_ndtr, roots_hermitenorm
from scipy.optimize import brentq
from IPython.display import display

N_SCENARIOS = 1_000
N_BONDS = 10
MATURITY_YEARS = 5
PAYMENTS_PER_YEAR = 4
FACE_VALUE = 10_000_000.0
ANNUAL_COUPON_RATE = 0.06
ANNUAL_DEFAULT_PROBABILITY = 0.04
LGD = 0.60
TARGET_DEFAULT_TIME_CORRELATION = 0.20
SEED = 6103

DT = 1 / PAYMENTS_PER_YEAR # time between payments in years
N_QUARTERS = MATURITY_YEARS * PAYMENTS_PER_YEAR
payment_times = np.arange(1, N_QUARTERS + 1) * DT
quarter_labels = [f"Q{q}" for q in range(1, N_QUARTERS + 1)]
bond_labels = [f"Bond {b}" for b in range(1, N_BONDS + 1)]
scenario_ids = pd.Index(range(1, N_SCENARIOS + 1), name="Scenario")
QUARTERLY_COUPON = FACE_VALUE * ANNUAL_COUPON_RATE * DT
RECOVERY = FACE_VALUE * (1 - LGD)

display(pd.Series({
    "Scenarios": N_SCENARIOS, "Bonds": N_BONDS,
    "Quarters": N_QUARTERS, "Face value per bond ($)": FACE_VALUE,
    "Quarterly coupon per bond ($)": QUARTERLY_COUPON,
    "Recovery per default ($)": RECOVERY,
}, name="Model inputs").to_frame())

,Model inputs
Scenarios,1000.0
Bonds,10.0
Quarters,20.0
Face value per bond ($),10000000.0
Quarterly coupon per bond ($),150000.0
Recovery per default ($),4000000.0


## 1. Convert annual default probability to a hazard rate

A **hazard rate** measures how quickly default can occur, **given that the bond has not already defaulted**. It describes default risk continuously through time.

It differs slightly from an annual default probability:

- **Annual default probability of 4%:** a surviving bond has a 4% chance of defaulting over the next full year.
- **Hazard rate:** the instantaneous rate of default among bonds that have survived so far. It is a rate per year, not itself a one-year probability.

With a constant hazard rate $\lambda$, survival through time $t$ is $S(t)=e^{-\lambda t}$, so

$$P(\text{default within }t\text{ years})=1-e^{-\lambda t}.$$

To make the one-year default probability equal to 4%, we solve:

$$0.04=1-e^{-\lambda}
\quad\Longrightarrow\quad
\lambda=-\ln(0.96)=0.0408219945\text{ per year}.$$

The notebook therefore uses a **hazard rate of about 4.0822% per year**, distinct from the 4% one-year probability. This lets us generate default times anywhere along the timeline—for example, at 0.6 years or 3.2 years—while preserving the required 4% annual default probability.

The conditional quarterly default probability is $1-e^{-\lambda/4}$, and the five-year cumulative default probability is $1-e^{-5\lambda}$.

In [2]:
hazard_rate = -np.log1p(-ANNUAL_DEFAULT_PROBABILITY)
quarterly_default_probability = -np.expm1(-hazard_rate * DT)
five_year_default_probability = -np.expm1(-hazard_rate * MATURITY_YEARS)
assert np.isclose(-np.expm1(-hazard_rate), ANNUAL_DEFAULT_PROBABILITY)
print(f"Annual hazard rate: {hazard_rate:.10f} ({hazard_rate:.6%})")
print(f"Conditional quarterly default probability: {quarterly_default_probability:.6%}")
print(f"Five-year cumulative default probability: {five_year_default_probability:.6%}")

Annual hazard rate: 0.0408219945 (4.082199%)
Conditional quarterly default probability: 1.015360%
Five-year cumulative default probability: 18.462730%


## 2. Calibrate correlation and generate default times

The requirement applies to **default times**, so we first find the normal-score correlation $\rho$ that produces a default-time correlation of 0.20. Then, for each scenario $s$, we draw a common standard normal factor $Z_s$ and independent standard normal bond factors $\epsilon_{s,i}$:

$$X_{s,i}=\sqrt{\rho}Z_s+\sqrt{1-\rho}\epsilon_{s,i},\qquad
U_{s,i}=\Phi(X_{s,i}),\qquad
\tau_{s,i}=-\frac{\ln U_{s,i}}{\lambda}.$$

**How calibration works.** Define $g(x)=-\ln\Phi(x)$. For a standard normal $X$, $g(X)$ has an exponential distribution with mean 1 and variance 1. Consequently,

$$\operatorname{Corr}(\tau_i,\tau_j)
=\operatorname{Corr}(g(X_i),g(X_j))
=E[g(X_i)g(X_j)]-1.$$

Dividing by the positive hazard rate does not change correlation. For a candidate $\rho$, we evaluate this expectation using two independent standard normals $A,B$, with $X_i=A$ and $X_j=\rho A+\sqrt{1-\rho^2}B$. This two-variable representation is used only to integrate a bond pair's distribution; the simulation below still uses the common-factor model for all 10 bonds.

Gauss–Hermite quadrature numerically evaluates the expectation, and a root solver finds the $\rho$ for which the resulting correlation is 0.20. We check the answer with a higher quadrature order. **Calibration uses the population distribution, independently of the 1,000 random scenarios.** It does not force every sample correlation to equal 0.20.

**Why keep the common-factor method?** All bond pairs share the same correlation. Cholesky decomposition of a normal-score correlation matrix with diagonal 1 and off-diagonal $\rho$ would produce the same joint normal distribution. Both methods need this calibration before transforming to default times.

The target applies to full default times, including those beyond five years, not to default indicators or times capped at maturity. Larger common-factor values produce shorter default times. The fixed seed preserves the base random draws for subsequent parameter comparisons.

In [3]:
def implied_default_time_correlation(rho, quadrature_order=48):
    """Population correlation of exponential times from correlated normals."""
    if not 0 <= rho <= 1:
        raise ValueError("The common-factor correlation must lie between 0 and 1.")
    nodes, weights = roots_hermitenorm(quadrature_order)
    weights = weights / np.sqrt(2 * np.pi)  # Standard normal expectation weights.
    x = nodes[:, None]
    y = rho * x + np.sqrt(1 - rho**2) * nodes[None, :]
    # log_ndtr evaluates log(Phi(x)) stably even far into the normal tail.
    g_x, g_y = -log_ndtr(x), -log_ndtr(y)
    expected_product = np.sum(weights[:, None] * weights[None, :] * g_x * g_y)
    return float(expected_product - 1.0)

if not 0 <= TARGET_DEFAULT_TIME_CORRELATION <= 1:
    raise ValueError("This common-factor model supports targets from 0 to 1.")
if TARGET_DEFAULT_TIME_CORRELATION in (0, 1):
    LATENT_RHO = float(TARGET_DEFAULT_TIME_CORRELATION)
else:
    LATENT_RHO = brentq(
        lambda rho: implied_default_time_correlation(rho) - TARGET_DEFAULT_TIME_CORRELATION,
        0.0, 1.0, xtol=1e-12,
    )

# Independently increase integration accuracy to check convergence.
verified_population_correlation = implied_default_time_correlation(LATENT_RHO, quadrature_order=80)
np.testing.assert_allclose(verified_population_correlation, TARGET_DEFAULT_TIME_CORRELATION,
                           rtol=0, atol=1e-9)
np.testing.assert_allclose(implied_default_time_correlation(0.0, 80), 0.0, rtol=0, atol=1e-9)
np.testing.assert_allclose(implied_default_time_correlation(1.0, 80), 1.0, rtol=0, atol=1e-9)
print(f"Target default-time correlation: {TARGET_DEFAULT_TIME_CORRELATION:.6f}")
print(f"Calibrated normal-score correlation: {LATENT_RHO:.10f}")
print(f"Verified population default-time correlation: {verified_population_correlation:.10f}")
print(f"For comparison, normal-score correlation 0.20 implies default-time correlation "
      f"{implied_default_time_correlation(0.20, 80):.6f}.")

Target default-time correlation: 0.200000
Calibrated normal-score correlation: 0.2332349314
Verified population default-time correlation: 0.2000000000
For comparison, normal-score correlation 0.20 implies default-time correlation 0.170302.


In [ ]:
rng = np.random.default_rng(SEED)
common_normals = rng.standard_normal((N_SCENARIOS, 1)) # common factor for all bonds in a scenario
idiosyncratic_normals = rng.standard_normal((N_SCENARIOS, N_BONDS)) # idiosyncratic factors for each bond in a scenario
latent_scores = np.sqrt(LATENT_RHO) * common_normals + np.sqrt(1 - LATENT_RHO) * idiosyncratic_normals
uniforms = ndtr(latent_scores)
# Guard against numerical rounding to exactly 0 or 1.
uniforms = np.clip(uniforms, np.finfo(float).tiny, np.nextafter(1.0, 0.0))
default_times = -np.log(uniforms) / hazard_rate
default_times_df = pd.DataFrame(default_times, index=scenario_ids, columns=bond_labels)

assert default_times.shape == (N_SCENARIOS, N_BONDS)
assert np.all(np.isfinite(default_times)) and np.all(default_times > 0)
print("Default times in years, first five scenarios:")
display(default_times_df.head().round(4))
print(f"Defaults within five years: {(default_times <= MATURITY_YEARS).sum():,} / {default_times.size:,}")

Default times in years, first five scenarios:


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10
Scenario,,,,,,,,,,
1,12.4212,47.1510,25.7353,59.5688,0.8596,11.7401,10.2084,16.5609,43.6182,27.3634
2,0.2842,0.4080,4.8380,4.3229,6.1740,13.5792,11.0591,52.9960,2.5031,1.0658
3,80.1860,0.4898,18.6178,9.6180,22.1169,5.1887,45.7410,53.1671,28.7273,57.2367
4,5.9021,47.8146,1.5034,0.4268,6.9459,5.2707,9.8119,4.8410,21.8049,24.7772
5,2.4485,1.7081,0.5795,9.4127,16.3852,17.4391,0.3564,1.7647,0.5230,20.6068


Defaults within five years: 1,839 / 10,000


## 3. Convert default times to quarterly bond cash flows

For payment date $t_q=q/4$, coupon $C=150{,}000$, face value $F=10{,}000{,}000$, and recovery $R=4{,}000{,}000$,

$$CF_{s,i,q}=
C\,\mathbf{1}_{\{\tau_{s,i}>t_q\}}
+R\,\mathbf{1}_{\{t_{q-1}<\tau_{s,i}\leq t_q\}}
+F\,\mathbf{1}_{\{q=20,\ \tau_{s,i}>5\}}.$$

The first term pays coupons while the bond survives; the second pays recovery only in the default quarter; the third repays principal at maturity only for surviving bonds. Recovery is allocated to a quarterly bucket without discounting. The project’s 9% YTM and 1% risk-free rate are inputs to later valuation, not to these undiscounted contractual cash flows.

In [5]:
def bond_cash_flow_components(times):
    """Map positive default times shaped (scenario, bond) to USD arrays.

    Each returned array has shape (scenario, bond, quarter).
    Positive infinity is allowed to represent no default in a test case.
    """
    times = np.asarray(times, dtype=float)
    if times.ndim != 2 or np.isnan(times).any() or np.any(times <= 0):
        raise ValueError("Default times must be a positive 2-D array; +inf is allowed.")
    tau = times[:, :, None]
    alive = tau > payment_times
    defaults_this_quarter = (tau > (payment_times - DT)) & (tau <= payment_times)
    coupons = QUARTERLY_COUPON * alive
    recoveries = RECOVERY * defaults_this_quarter
    principal = np.zeros_like(coupons)
    principal[:, :, -1] = FACE_VALUE * (times > MATURITY_YEARS)
    return coupons, recoveries, principal

coupon_cash_flows, recovery_cash_flows, principal_cash_flows = bond_cash_flow_components(default_times)
bond_cash_flows = coupon_cash_flows + recovery_cash_flows + principal_cash_flows
assert bond_cash_flows.shape == (N_SCENARIOS, N_BONDS, N_QUARTERS)
assert np.all(bond_cash_flows >= 0)
np.testing.assert_allclose(
    recovery_cash_flows.sum(axis=2), RECOVERY * (default_times <= MATURITY_YEARS)
)
assert np.all(bond_cash_flows[default_times[:, :, None] <= (payment_times - DT)] == 0)
print("Bond cash flow array shape:", bond_cash_flows.shape)

Bond cash flow array shape: (1000, 10, 20)


## 4. Sum the 10 bonds and inspect a scenario

`collateral_cash_flows[s, q]` is the sum across all bonds for one scenario and quarter. Scenario labels are 1–1,000; NumPy indices are 0–999. The inspection table shows bond cash flows and separate collateral coupon, recovery, and principal totals. All table values below are in **millions of dollars**.

In [6]:
collateral_cash_flows = bond_cash_flows.sum(axis=1)
collateral_cash_flows_df = pd.DataFrame(
    collateral_cash_flows, index=scenario_ids, columns=quarter_labels
)
assert collateral_cash_flows.shape == (N_SCENARIOS, N_QUARTERS)

def inspect_scenario(scenario_id):
    if not isinstance(scenario_id, (int, np.integer)) or not 1 <= scenario_id <= N_SCENARIOS:
        raise ValueError(f"Choose an integer scenario ID from 1 to {N_SCENARIOS}.")
    s = scenario_id - 1
    default_quarters = [
        int(np.searchsorted(payment_times, t, side="left") + 1)
        if t <= MATURITY_YEARS else None for t in default_times[s]
    ]
    details = pd.DataFrame({
        "Default time (years)": default_times[s],
        "Default quarter": pd.array(default_quarters, dtype="Int64"),
        "Survives maturity": default_times[s] > MATURITY_YEARS,
    }, index=bond_labels)
    cash = pd.DataFrame(bond_cash_flows[s].T / 1e6, index=quarter_labels, columns=bond_labels)
    cash["Coupons"] = coupon_cash_flows[s].sum(axis=0) / 1e6
    cash["Recoveries"] = recovery_cash_flows[s].sum(axis=0) / 1e6
    cash["Principal"] = principal_cash_flows[s].sum(axis=0) / 1e6
    cash["Collateral total"] = collateral_cash_flows[s] / 1e6
    print(f"Scenario {scenario_id}; cash flows in $ millions; <NA> means no default within five years.")
    display(details.round(4))
    display(cash.round(4))
    return details, cash

SCENARIO_ID = 1  # Change to any integer from 1 to 1000; no new random draws are needed.
scenario_details, scenario_cash_flows = inspect_scenario(SCENARIO_ID)

Scenario 1; cash flows in $ millions; <NA> means no default within five years.


,Default time (years),Default quarter,Survives maturity
Bond 1,12.4212,<NA>,True
Bond 2,47.1510,<NA>,True
Bond 3,25.7353,<NA>,True
Bond 4,59.5688,<NA>,True
Bond 5,0.8596,4,False
Bond 6,11.7401,<NA>,True
Bond 7,10.2084,<NA>,True
Bond 8,16.5609,<NA>,True
Bond 9,43.6182,<NA>,True
Bond 10,27.3634,<NA>,True


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10,Coupons,Recoveries,Principal,Collateral total
Q1,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,1.50,0.0,0.0,1.50
Q2,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,1.50,0.0,0.0,1.50
Q3,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,1.50,0.0,0.0,1.50
Q4,0.15,0.15,0.15,0.15,4.00,0.15,0.15,0.15,0.15,0.15,1.35,4.0,0.0,5.35
Q5,0.15,0.15,0.15,0.15,0.00,0.15,0.15,0.15,0.15,0.15,1.35,0.0,0.0,1.35
Q6,0.15,0.15,0.15,0.15,0.00,0.15,0.15,0.15,0.15,0.15,1.35,0.0,0.0,1.35
Q7,0.15,0.15,0.15,0.15,0.00,0.15,0.15,0.15,0.15,0.15,1.35,0.0,0.0,1.35
Q8,0.15,0.15,0.15,0.15,0.00,0.15,0.15,0.15,0.15,0.15,1.35,0.0,0.0,1.35
Q9,0.15,0.15,0.15,0.15,0.00,0.15,0.15,0.15,0.15,0.15,1.35,0.0,0.0,1.35
Q10,0.15,0.15,0.15,0.15,0.00,0.15,0.15,0.15,0.15,0.15,1.35,0.0,0.0,1.35


## 5. Validate cash flows by hand

The following deliberately constructed scenarios give transparent checks independent of the random draws. Amounts are in $ millions.

| Scenario | Hand calculation | Total over 20 quarters |
|---|---|---:|
| No defaults | Q1–Q19: 10 × 0.15 = 1.50 each; Q20: 10 × 10.15 = 101.50 | 130.00 |
| Bond 1 defaults at 0.10 years | Q1: 9 × 0.15 + 4 = 5.35; Q2–Q19: 1.35 each; Q20: 9 × 10.15 = 91.35 | 121.00 |
| Bond 1 defaults at 0.60 years | Q1–Q2: 1.50 each; Q3: 1.35 + 4 = 5.35; Q4–Q19: 1.35 each; Q20: 91.35 | 121.30 |
| All bonds default at 4.90 years | Q1–Q19: 1.50 each; Q20: 10 × 4 = 40.00 | 68.50 |

Boundary checks also verify that default at exactly 0.25 years replaces the Q1 coupon with recovery, and default at exactly five years replaces the final coupon and principal with recovery.

In [7]:
manual_times = np.full((4, N_BONDS), np.inf)
manual_times[1, 0] = 0.10
manual_times[2, 0] = 0.60
manual_times[3, :] = 4.90
manual_actual = sum(bond_cash_flow_components(manual_times)).sum(axis=1)

# Explicit expected schedules from the hand calculations above, in dollars.
manual_expected = np.array([
    [1.50] * 19 + [101.50],
    [5.35] + [1.35] * 18 + [91.35],
    [1.50, 1.50, 5.35] + [1.35] * 16 + [91.35],
    [1.50] * 19 + [40.00],
]) * 1e6
np.testing.assert_allclose(manual_actual, manual_expected, rtol=0, atol=1e-6)
boundary_actual = sum(bond_cash_flow_components(np.array([[0.25, 5.0, 5.000001]])))
boundary_expected = np.array([
    [4.00] + [0.00] * 19,
    [0.15] * 19 + [4.00],
    [0.15] * 19 + [10.15],
]) * 1e6
np.testing.assert_allclose(boundary_actual[0], boundary_expected, rtol=0, atol=1e-6)
display(pd.DataFrame({
    "Expected total ($m)": manual_expected.sum(axis=1) / 1e6,
    "Simulated total ($m)": manual_actual.sum(axis=1) / 1e6,
    "All 20 quarters match": np.isclose(manual_actual, manual_expected).all(axis=1),
}, index=["No defaults", "One default in Q1", "One default in Q3", "All default in Q20"]))
print("All manual schedules and payment-date boundary checks passed.")

,Expected total ($m),Simulated total ($m),All 20 quarters match
No defaults,130.0,130.0,True
One default in Q1,121.0,121.0,True
One default in Q3,121.3,121.3,True
All default in Q20,68.5,68.5,True


All manual schedules and payment-date boundary checks passed.


### Cross-check three actual simulated scenarios

For three scenarios, the next cell independently walks through each bond and each payment date. It also counts surviving bonds and new defaults each quarter to verify the arithmetic: coupon cash = survivor count × $150,000; recovery cash = new defaults × $4 million; final principal = maturity survivors × $10 million. This catches incorrect bond aggregation and recovery timing.

In [8]:
def cash_flows_by_loop(times):
    result = np.zeros((len(times), N_QUARTERS))
    for b, tau in enumerate(times):
        for q, t in enumerate(payment_times):
            if tau <= t:
                result[b, q] = RECOVERY
                break
            result[b, q] = QUARTERLY_COUPON
            if q == N_QUARTERS - 1:
                result[b, q] += FACE_VALUE
    return result

default_counts = (default_times <= MATURITY_YEARS).sum(axis=1)
selected_indices = np.unique([0, int(np.argmin(default_counts)), int(np.argmax(default_counts))]).tolist()
for candidate in range(N_SCENARIOS):
    if len(selected_indices) >= 3:
        break
    if candidate not in selected_indices:
        selected_indices.append(candidate)

audit_rows = []
for s in selected_indices:
    reference = cash_flows_by_loop(default_times[s])
    np.testing.assert_allclose(bond_cash_flows[s], reference, rtol=0, atol=1e-6)
    np.testing.assert_allclose(collateral_cash_flows[s], reference.sum(axis=0), rtol=0, atol=1e-6)
    for q, t in enumerate(payment_times):
        surviving = int(np.sum(default_times[s] > t))
        new_defaults = int(np.sum((default_times[s] > t - DT) & (default_times[s] <= t)))
        hand_total = surviving * QUARTERLY_COUPON + new_defaults * RECOVERY
        if q == N_QUARTERS - 1:
            hand_total += surviving * FACE_VALUE
        np.testing.assert_allclose(hand_total, collateral_cash_flows[s, q], rtol=0, atol=1e-6)
        if q in (0, N_QUARTERS - 1) or new_defaults:
            audit_rows.append([s + 1, q + 1, surviving, new_defaults, hand_total / 1e6, collateral_cash_flows[s, q] / 1e6])

display(pd.DataFrame(audit_rows, columns=[
    "Scenario", "Quarter", "Survivors", "New defaults", "Hand total ($m)", "Model total ($m)"
]))
print(f"All bond and collateral cash flows checked for scenarios {[s + 1 for s in selected_indices]}.")

,Scenario,Quarter,Survivors,New defaults,Hand total ($m),Model total ($m)
0,1,1,10,0,1.50,1.50
1,1,4,9,1,5.35,5.35
2,1,20,9,0,91.35,91.35
3,9,1,10,0,1.50,1.50
4,9,20,10,0,101.50,101.50
5,62,1,10,0,1.50,1.50
6,62,2,9,1,5.35,5.35
7,62,3,8,1,5.20,5.20
8,62,4,7,1,5.05,5.05
9,62,6,6,1,4.90,4.90


All bond and collateral cash flows checked for scenarios [1, 9, 62].


## 6. Simulation diagnostics and handoff

As a further check, expected collateral cash flow at quarter $q$ is

$$E[CF_q]=10\left[C S(t_q)+R\{S(t_{q-1})-S(t_q)\}
+F\,\mathbf{1}_{\{q=20\}}S(5)\right].$$

Correlation affects dispersion but not this expectation. The sample means will differ from the theoretical values because there are only 1,000 scenarios. Standard errors are computed across independent **scenario totals**, preserving dependence between bonds within a scenario. Correlation diagnostics are descriptive, not requirements that a finite sample equal its population target exactly.

In [9]:
survival = np.exp(-hazard_rate * payment_times)
previous_survival = np.exp(-hazard_rate * (payment_times - DT))
expected_collateral = N_BONDS * (
    QUARTERLY_COUPON * survival + RECOVERY * (previous_survival - survival)
)
expected_collateral[-1] += N_BONDS * FACE_VALUE * survival[-1]
simulation_mean = collateral_cash_flows.mean(axis=0)
simulation_se = collateral_cash_flows.std(axis=0, ddof=1) / np.sqrt(N_SCENARIOS)
diagnostics = pd.DataFrame({
    "Theoretical mean ($m)": expected_collateral / 1e6,
    "Simulation mean ($m)": simulation_mean / 1e6,
    "Monte Carlo SE ($m)": simulation_se / 1e6,
    "Difference / SE": np.divide(simulation_mean - expected_collateral, simulation_se,
                                 out=np.full(N_QUARTERS, np.nan), where=simulation_se > 0),
}, index=quarter_labels)
display(diagnostics.round(4))

pairs = np.triu_indices(N_BONDS, k=1)
mean_latent_correlation = np.corrcoef(latent_scores, rowvar=False)[pairs].mean()
default_time_correlation_matrix = np.corrcoef(default_times, rowvar=False)
pairwise_time_correlations = default_time_correlation_matrix[pairs]
mean_time_correlation = pairwise_time_correlations.mean()
print(f"Average sample latent-score correlation: {mean_latent_correlation:.4f} (calibrated target {LATENT_RHO:.4f})")
print(f"Population default-time correlation: {verified_population_correlation:.6f} "
      f"(target {TARGET_DEFAULT_TIME_CORRELATION:.2f})")
print(f"Average sample default-time correlation: {mean_time_correlation:.4f}")
print(f"Range across the 45 sample bond-pair correlations: "
      f"{pairwise_time_correlations.min():.4f} to {pairwise_time_correlations.max():.4f}")
print("Sample correlations vary because there are 1,000 scenarios; calibration targets the population.")
display(pd.DataFrame(default_time_correlation_matrix, index=bond_labels, columns=bond_labels).round(3))
print(f"Five-year default frequency: {default_counts.mean() / N_BONDS:.4%}")
print(f"Theoretical five-year default probability: {five_year_default_probability:.4%}")
print(f"Mean number of defaults per scenario: {default_counts.mean():.3f}")
print(f"Mean total collateral cash ($m): {collateral_cash_flows.sum(axis=1).mean() / 1e6:.4f}")
print("Validation complete: all deterministic cash-flow assertions passed.")

,Theoretical mean ($m),Simulation mean ($m),Monte Carlo SE ($m),Difference / SE
Q1,1.8909,1.8888,0.0444,-0.0465
Q2,1.8717,1.8891,0.0403,0.4310
Q3,1.8527,1.8502,0.0392,-0.0627
Q4,1.8339,1.8238,0.0384,-0.2615
Q5,1.8153,1.8441,0.0416,0.6933
Q6,1.7968,1.8130,0.0400,0.4031
Q7,1.7786,1.7170,0.0341,-1.8097
Q8,1.7605,1.6857,0.0333,-2.2471
Q9,1.7427,1.7399,0.0368,-0.0752
Q10,1.7250,1.7838,0.0409,1.4396


Average sample latent-score correlation: 0.2328 (calibrated target 0.2332)
Population default-time correlation: 0.200000 (target 0.20)
Average sample default-time correlation: 0.2033
Range across the 45 sample bond-pair correlations: 0.1518 to 0.2554
Sample correlations vary because there are 1,000 scenarios; calibration targets the population.


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10
Bond 1,1.000,0.243,0.185,0.185,0.255,0.218,0.199,0.208,0.163,0.184
Bond 2,0.243,1.000,0.221,0.185,0.198,0.247,0.152,0.250,0.159,0.206
Bond 3,0.185,0.221,1.000,0.239,0.200,0.222,0.168,0.212,0.161,0.201
Bond 4,0.185,0.185,0.239,1.000,0.225,0.248,0.206,0.215,0.183,0.196
Bond 5,0.255,0.198,0.200,0.225,1.000,0.211,0.154,0.164,0.189,0.238
Bond 6,0.218,0.247,0.222,0.248,0.211,1.000,0.243,0.249,0.179,0.178
Bond 7,0.199,0.152,0.168,0.206,0.154,0.243,1.000,0.163,0.240,0.208
Bond 8,0.208,0.250,0.212,0.215,0.164,0.249,0.163,1.000,0.205,0.179
Bond 9,0.163,0.159,0.161,0.183,0.189,0.179,0.240,0.205,1.000,0.215
Bond 10,0.184,0.206,0.201,0.196,0.238,0.178,0.208,0.179,0.215,1.000


Five-year default frequency: 18.3900%
Theoretical five-year default probability: 18.4627%
Mean number of defaults per scenario: 1.839
Mean total collateral cash ($m): 115.9786
Validation complete: all deterministic cash-flow assertions passed.


### Outputs for the next project section

| Variable | Shape / labels | Contents |
|---|---|---|
| `LATENT_RHO` | Scalar | Calibrated normal-score correlation for default-time target 0.20 |
| `common_normals`, `idiosyncratic_normals` | (1000, 1), (1000, 10) | Fixed base random draws for reuse |
| `default_times` | (1000, 10) | Default times in years |
| `default_times_df` | Scenarios 1–1000 × Bonds 1–10 | Labeled default times |
| `bond_cash_flows` | (1000, 10, 20) | Individual bond cash flows in dollars |
| `coupon_cash_flows`, `recovery_cash_flows`, `principal_cash_flows` | (1000, 10, 20) each | Separate cash flow components |
| `collateral_cash_flows` | (1000, 20) | Dollar cash available to the waterfall |
| `collateral_cash_flows_df` | Scenarios 1–1000 × Q1–Q20 | Labeled collateral cash flows |

All arrays contain undiscounted inflows. There is no time-zero purchase price in these schedules. Later waterfall calculations can consume `collateral_cash_flows` directly.